# Chapter 17 — It Doesn't Hurt to Trie

**Source:** Wengrow, *A Common-Sense Guide to Data Structures and Algorithms*, Ch. 17

## The problem: autocomplete
You type `"catn"` → the phone suggests `"catnip"`, `"catnap"`. Where are the words stored?

| Structure | Find all words starting with "catn" |
|---|---|
| Unsorted array | O(N): check every word in the dictionary |
| Hash table | ❌ hashes the **whole** word, so there's no key `"catn"` to look up |
| Ordered array + binary search | O(log N) |
| **Trie** | **O(K)**, K = length of what you typed, *regardless of dictionary size* |

**Trie** comes from re*trie*val. It's pronounced "try" so it isn't confused with "tree". It's also called a **prefix tree**.
Uses: autocomplete, autocorrect, and also IP addresses and phone numbers (anything with shared prefixes).

## 1. The trie node

A trie is **not** a binary tree: a node can have **any number** of children.
In this implementation, each node is just a **hash table (dict)**:
- **key** = a single character
- **value** = the child `TrieNode`

Each word is stored as a **path of characters** from the root. Words with a common prefix **share** the start of the path:
`"ace"` and `"act"` share the `a → c` nodes, then branch into `e` and `t`.

### The `"*"` end-of-word marker
A `"*"` key marks "a complete word ends here".
Why it's needed: `"bat"` and `"batter"`. The `t` node after `b-a` needs to say both "a word ends here" (`*`) **and** "keep going" (`t`). Without `*`, you couldn't tell that `"bat"` is a word and not just a prefix of `"batter"`.

          root
           |
           b
           |
           a
           |
           t
         /   \
        *     t        <- "bat" ends here; "batter" continues
              |
              e
              |
              r
              |
              *

This is your **dict of dicts** knowledge from Fluent Python (Week 1), now used as a tree.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}          # char -> TrieNode   (or "*" -> value)


class Trie:
    def __init__(self):
        self.root = TrieNode()      # empty root

## 2. Search: O(K)

Algorithm (prefix search):
1. `current` = root.
2. For each character of the search string: if `current` has that character as a child key → move to that child. Otherwise → return `None` (not in the trie).
3. If you get through all the characters → found. **Return the node you ended on** (not just `True`). That node is reused by autocomplete.

This finds **prefixes**, so a complete word is found too. To check "is this a *complete word*?", also check that `"*"` is among the final node's children.

### Why O(K) and not O(N) or O(log N)?
- Each character = **one dict lookup = O(1)**. So the steps = number of characters in the search string.
- **K** = the length of the search string. We use a new letter because N means "amount of data".
- The important part: searching "cat" takes **3 steps whether the trie holds 10 words or 1 million**. Speed depends on the *input*, not on the *amount of stored data*, so it's almost as good as constant time.

In [ ]:
def search(self, word):
    current = self.root
    for char in word:
        if char in current.children:          # (book: .get(char) -- same idea)
            current = current.children[char]
        else:
            return None                       # dead end -> not in trie
    return current                            # node of the last character

Trie.search = search

## 3. Insertion: O(K)

Same walk as search, but when a character is **missing**, **create** a new node for it instead of giving up.
After the last character, add `"*"` to mark the end of the word.
Steps: K + 1 → **O(K)**.

Example: insert `"can"` into a trie that already has `"cat"` → reuse `c`, reuse `a`, **create** `n`, add `*`.

I added an optional `value` parameter. It's stored as the value of the `"*"` key (default `None`), and it's used for the popularity feature at the end.

In [ ]:
def insert(self, word, value=None):
    current = self.root
    for char in word:
        if char not in current.children:
            current.children[char] = TrieNode()   # create the missing node
        current = current.children[char]          # move down
    current.children["*"] = value                 # end-of-word marker

Trie.insert = insert


t = Trie()
for w in ["ace", "act", "bad", "bake", "bat", "batter", "cab", "cat", "catnap", "catnip"]:
    t.insert(w)

node = t.search("cat")
print(node is not None)               # True  -> "cat" is at least a prefix
print("*" in node.children)           # True  -> and a complete word
print("*" in t.search("catn").children)   # False -> "catn" is only a prefix
print(t.search("dog"))                # None

## 4. Collecting all words (recursive)

`collect_all_words(node)` returns every word below a given node:
- Loop over the node's children.
- If the key is `"*"` → **base case**: the word built so far is complete, so add it to the list.
- Otherwise → **recurse** into the child with `word + key` (the word grows by one letter).

Two things travel down the recursion:
- `word` (a **string**): strings are **immutable**, so each call has its *own* `word`. When the recursion returns, the caller still has its shorter version (`"ca"`, not `"can"`).
- `words` (a **list**): one **shared, mutable** object. Every call appends to the *same* list, exactly like the `memo` dict in Chapter 12.

### ⚠️ Bug in the book's Python: mutable default argument
The book writes `def collectAllWords(self, node=None, word="", words=[])`.
The `[]` default is created **once** and shared across **all calls of the function**, so the second time you call it, it still contains the words from the first call. Fix: `words=None`, then create a new list inside. (This is the gotcha from the Ch. 12 memo notes.)

In [ ]:
# Demonstrating the book's bug:
def collect_buggy(self, node=None, word="", words=[]):     # <- mutable default!
    current = node or self.root
    for key, child in current.children.items():
        if key == "*":
            words.append(word)
        else:
            collect_buggy(self, child, word + key, words)
    return words

small = Trie()
for w in ["can", "cat"]:
    small.insert(w)
print(collect_buggy(small))     # ['can', 'cat']
print(collect_buggy(small))     # ['can', 'cat', 'can', 'cat']  <- leftover from the 1st call!


# Fixed version:
def collect_all_words(self, node=None, word="", words=None):
    if words is None:
        words = []                                  # fresh list per top-level call
    current = node or self.root
    for key, child in current.children.items():
        if key == "*":
            words.append(word)                      # base case: complete word
        else:
            self.collect_all_words(child, word + key, words)
    return words

Trie.collect_all_words = collect_all_words

print(t.collect_all_words())
print(t.collect_all_words())    # same result both times

## 5. Autocomplete = search + collect

1. `search(prefix)` → the node of the prefix's last character (or `None`).
2. `collect_all_words(that_node)` → all **endings** that complete it.

The book returns just the endings (`"nap"`, `"nip"` for `"catn"`). I add the prefix back on so you see full words.

## 6. Better autocomplete: tries with values
Showing 16 suggestions is useless; you want the **most popular** ones.
Instead of `None`, store a **popularity score** as the value of the `"*"` key:
`"ball"` → 10, `"balance"` → 9, `"bald"` → 7, `"balter"` (= dancing clumsily) → 1.
Collect the `(word, score)` pairs, sort by score, and show the top few.

In [ ]:
def autocomplete(self, prefix):
    node = self.search(prefix)
    if node is None:
        return None
    return [prefix + ending for ending in self.collect_all_words(node)]

Trie.autocomplete = autocomplete

print(t.autocomplete("catn"))   # ['catnap', 'catnip']
print(t.autocomplete("ba"))     # ['bad', 'bake', 'bat', 'batter']
print(t.autocomplete("xyz"))    # None


# --- with popularity scores ---
def collect_with_scores(node, word="", out=None):
    if out is None:
        out = []
    for key, child in node.children.items():
        if key == "*":
            out.append((word, child))            # child = the stored score here
        else:
            collect_with_scores(child, word + key, out)
    return out

pop = Trie()
for word, score in [("ball", 10), ("balance", 9), ("bald", 7), ("balter", 1)]:
    pop.insert(word, score)

prefix = "bal"
suggestions = collect_with_scores(pop.search(prefix))
top3 = sorted(suggestions, key=lambda pair: pair[1], reverse=True)[:3]
print([prefix + ending for ending, score in top3])   # ball, balance, bald

## Summary

| Operation | Trie |
|---|---|
| Search (word or prefix) | **O(K)** |
| Insert | **O(K)** |
| Autocomplete | O(K) to find the prefix + the size of the subtree below it |

- Trie = tree where each node is a **dict of character → child node**. Words = paths; shared prefixes = shared nodes.
- `"*"` marks the end of a word, and its value can carry data (e.g. popularity).
- O(K) depends only on the length of the input, **not** on how many words are stored.
- Trees covered so far: **BST** (ordered data, frequent changes), **heap** (priority queue), **trie** (text prefixes). Next chapter: **graphs**, the last data structure in the book.
- Watch out for **mutable default arguments** in Python recursion (`words=[]`). Use `None`.

## Exercises (book solutions: p. 459)

### Exercise 1: read a trie
**Input:** the trie below, written top-down (indentation = child; `*` = end of a word; `{*, l, n}` means that node has three keys).

    root
    ├── t
    │   ├── a
    │   │   ├── g ── *
    │   │   ├── n ── {*, k}
    │   │   │              └── k ── *
    │   │   └── p ── *
    │   └── o
    │       ├── d ── a ── y ── *
    │       └── t ── a ── l ── *
    └── w
        └── e ── {*, l, n}
                     ├── l ── l ── *
                     └── n ── t ── *

**Task:** list every word stored in it (follow each path from the root to a `*`). Then check with the code.

### Exercise 2: draw a trie
**Input:** the words `"get", "go", "got", "gotten", "hall", "ham", "hammer", "hill", "zebra"`.
**Task:** draw the trie on paper. Watch where words **share prefixes** (`g`, `go`, `got`, `ha`, `ham` …) and where `*` is needed *in the middle* of a path (e.g. `"got"` inside `"gotten"`). Check with `show_trie` afterwards.

### Exercise 3: print every key
**Input:** a trie's root node (e.g. `t.root`).
**Task:** write `print_all_keys(node)` that visits **every** node and prints **every key, including the `"*"` keys**. (Recursion, like `collect_all_words`. Careful: the value of `"*"` is not a `TrieNode`, so don't recurse into it.)

### Exercise 4: autocorrect (the challenging one)
**Input:** a trie and a string the user typed.
**Task:** write `autocorrect(trie, text)`:
- If `text` is found in the trie (even as a prefix) → return `text` unchanged.
- Otherwise → return a word from the trie that shares the **longest possible prefix** with `text`.

Examples with the trie `{"cat", "catnap", "catnip"}`:
- `"catnar"` → `"catnap"` (shares `"catna"`, 5 chars; `"catnip"` only shares `"catn"`)
- `"caxasfdij"` → any of `"cat"`, `"catnap"`, `"catnip"` (all share `"ca"`)
- `"cat"` → `"cat"` (found, return as is)

Hint: walk down the trie like `search`, but **remember how far you got** before the dead end. Then use a function you already have, starting from that node.